In [10]:
import pandas as pd
import numpy as np
import json
import os
import joblib
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler

RAW_PATH = os.path.join("..", "..", "data", "raw", "travel.csv")
PROCESSED_DIR = os.path.join("..", "..", "data", "processed")
ARTIFACTS_DIR = os.path.join("..", "artifacts")
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

## 1. Load & clean raw data

In [11]:
df = pd.read_csv(RAW_PATH)

text_cols = df.select_dtypes(include=["object", "string"]).columns
for col in text_cols:
    df[col] = df[col].astype("string").str.strip().astype(object)

print(f"Loaded {len(df)} rows, {df.shape[1]} columns.")
df.head()

Loaded 1163 rows, 27 columns.


,place_id,Place_Name,City,State,Zone,Place_Type,Interest_Tags,Suitable_For,Entry_Fee_INR,Activity_Cost_Min,...,Popularity_Rating,Number_Of_Reviewpoints,Best_Month_Start,Best_Month_End,Terrain_Type,Cluster_ID,Hidden_Gem_Score,Is_Hidden_Gem,Difficulty_Level,Typical_Duration
0,1,Dal Lake,Srinagar,Jammu And Kashmir,North,Lake,"Nature,Scenic,Relaxation","Couples,Family,Solo",0,500,...,4.3,4300,4,10,Valley,JAM-110,47.2,False,Easy,Full-day (4-8 hrs)
1,2,Shalimar Bagh,Srinagar,Jammu And Kashmir,North,Garden,"Historical,Cultural,Nature","Couples,Family,Solo",40,0,...,5.0,1000,4,10,Valley,JAM-110,92.4,True,Easy,Short (1-2 hrs)
2,3,Sonamarg,Ganderbal,Jammu And Kashmir,North,Valley,"Scenic,Adventure,Nature","Couples,Family,Friends,Solo",0,500,...,5.0,10000,5,9,Valley,JAM-110,81.4,True,Moderate,Full-day (4-8 hrs)
3,4,Gulmarg,Baramulla,Jammu And Kashmir,North,Hill Station,"Adventure,Scenic,Winter Sports","Couples,Family,Friends,Solo",0,700,...,5.0,10000,3,6,Mountain,JAM-110,81.4,True,Moderate,Full-day (4-8 hrs)
4,5,Noori Chamb Waterfall,Poonch,Jammu And Kashmir,North,Waterfall,"Nature,Scenic,Historic","Couples,Solo,Friends",0,0,...,4.0,700,4,10,Mountain,JAM-112,39.5,False,Moderate,Half-day (2-4 hrs)


## 2. Multi-hot encode Interest_Tags and Suitable_For
Turns `"Nature, Scenic, Adventure"` into separate binary columns the ML ranking model can use directly.

In [12]:
def split_multivalue(series):
    return series.apply(lambda s: [t.strip() for t in str(s).split(",") if t.strip()])

interest_lists = split_multivalue(df["Interest_Tags"])
suitable_lists = split_multivalue(df["Suitable_For"])

interest_mlb = MultiLabelBinarizer()
interest_encoded = interest_mlb.fit_transform(interest_lists)
interest_df = pd.DataFrame(
    interest_encoded,
    columns=[f"interest_{c.lower().replace(' ', '_')}" for c in interest_mlb.classes_],
    index=df.index,
)

suitable_mlb = MultiLabelBinarizer()
suitable_encoded = suitable_mlb.fit_transform(suitable_lists)
suitable_df = pd.DataFrame(
    suitable_encoded,
    columns=[f"suitable_{c.lower().replace(' ', '_')}" for c in suitable_mlb.classes_],
    index=df.index,
)

joblib.dump(interest_mlb, os.path.join(ARTIFACTS_DIR, "interest_mlb.pkl"))
joblib.dump(suitable_mlb, os.path.join(ARTIFACTS_DIR, "suitable_mlb.pkl"))

df = pd.concat([df, interest_df, suitable_df], axis=1)
print(f"Added {interest_df.shape[1]} interest columns and {suitable_df.shape[1]} suitable_for columns.")
df[list(suitable_df.columns)].sum()

Added 152 interest columns and 14 suitable_for columns.


suitable_adventure                  60
suitable_children                    8
suitable_couples                   362
suitable_family                    689
suitable_friends                   561
suitable_international_tourists      2
suitable_photographers              92
suitable_pilgrims                   90
suitable_professionals               1
suitable_shoppers                    1
suitable_solo                      398
suitable_spiritual                   1
suitable_students                  121
suitable_tourists                  154
dtype: int64

## 3. Ordinal-encode Difficulty_Level
Easy=1, Moderate=2, Strenuous=3 — used directly by the rule-based age/health safety filter (e.g. block Strenuous for 60+).

In [13]:
difficulty_map = {"Easy": 1, "Moderate": 2, "Strenuous": 3}
df["difficulty_score"] = df["Difficulty_Level"].map(difficulty_map)

if df["difficulty_score"].isnull().any():
    unmapped = df.loc[df["difficulty_score"].isnull(), "Difficulty_Level"].unique()
    raise ValueError(f"Unmapped Difficulty_Level values found: {unmapped}")

with open(os.path.join(ARTIFACTS_DIR, "difficulty_map.json"), "w") as f:
    json.dump(difficulty_map, f, indent=2)

df["difficulty_score"].value_counts()

difficulty_score
2    690
1    447
3     26
Name: count, dtype: int64

## 4. Compute per-place costs
EDA showed `*_Min` and `*_Max` are ~0.9+ correlated, so we keep one averaged feature per cost component for the ML ranker, while still keeping `total_daily_cost_min`/`max` for the exact rule-based budget filter.

In [14]:
df["activity_cost_avg"] = (df["Activity_Cost_Min"] + df["Activity_Cost_Max"]) / 2
df["hotel_cost_avg"] = (df["HotelcostpernightINR_Min"] + df["HotelcostpernightINR_Max"]) / 2
df["food_cost_avg"] = (df["FoodcostperdayINR_Min"] + df["FoodcostperdayINR_Max"]) / 2

df["total_daily_cost_min"] = (
    df["Entry_Fee_INR"] + df["Activity_Cost_Min"] +
    df["HotelcostpernightINR_Min"] + df["FoodcostperdayINR_Min"]
)
df["total_daily_cost_max"] = (
    df["Entry_Fee_INR"] + df["Activity_Cost_Max"] +
    df["HotelcostpernightINR_Max"] + df["FoodcostperdayINR_Max"]
)
df["total_daily_cost_avg"] = (df["total_daily_cost_min"] + df["total_daily_cost_max"]) / 2

df[["total_daily_cost_min", "total_daily_cost_max", "total_daily_cost_avg"]].describe()

,total_daily_cost_min,total_daily_cost_max,total_daily_cost_avg
count,1163.000000,1163.000000,1163.000000
mean,2231.711092,6990.610490,4611.160791
std,766.021415,2667.041391,1665.770739
min,1060.000000,3250.000000,2350.000000
25%,1750.000000,5165.000000,3525.000000
50%,2100.000000,6300.000000,4230.000000
75%,2500.000000,7900.000000,5100.000000
max,9800.000000,26000.000000,17900.000000


## 5. Build state-level budget lookup table
Used by the rule layer to hard-filter states before the ML ranker runs.

In [15]:
state_budget_lookup = (
    df.groupby("State")[["total_daily_cost_min", "total_daily_cost_max", "total_daily_cost_avg"]]
    .mean()
    .round(0)
    .reset_index()
    .sort_values("total_daily_cost_avg")
)
state_budget_lookup.to_csv(os.path.join(ARTIFACTS_DIR, "state_budget_lookup.csv"), index=False)

print("Cheapest 5 states:")
state_budget_lookup.head()

Cheapest 5 states:


,State,total_daily_cost_min,total_daily_cost_max,total_daily_cost_avg
32,Tripura,2085.0,4111.0,3098.0
23,Mizoram,1870.0,4376.0,3123.0
3,Bihar,1679.0,4950.0,3314.0
24,Nagaland,2201.0,4431.0,3316.0
14,Jharkhand,1890.0,5173.0,3531.0


## 6. Min-max scale ranking features
Scaled 0-1 versions of Popularity_Rating, Hidden_Gem_Score, Number_Of_Reviewpoints for the ML ranker (raw columns kept too).

In [16]:
scale_cols = ["Popularity_Rating", "Hidden_Gem_Score", "Number_Of_Reviewpoints"]
scaler = MinMaxScaler()
scaled = scaler.fit_transform(df[scale_cols])
for i, col in enumerate(scale_cols):
    df[f"{col.lower()}_scaled"] = scaled[:, i]

joblib.dump(scaler, os.path.join(ARTIFACTS_DIR, "scaler.pkl"))
df[[f"{c.lower()}_scaled" for c in scale_cols]].describe()

,popularity_rating_scaled,hidden_gem_score_scaled,number_of_reviewpoints_scaled
count,1163.000000,1163.000000,1163.000000
mean,0.556476,0.472133,0.019829
std,0.166117,0.118489,0.043256
min,0.000000,0.000000,0.000000
25%,0.454545,0.393586,0.003112
50%,0.545455,0.473761,0.007557
75%,0.636364,0.551020,0.019782
max,1.000000,1.000000,1.000000


## 7. Expand seasonality
Turns `Best_Month_Start`/`Best_Month_End` into an explicit list of in-season months (handles wrap-around, e.g. Nov→Feb).

In [17]:
def months_open(row):
    s, e = int(row["Best_Month_Start"]), int(row["Best_Month_End"])
    if s <= e:
        return list(range(s, e + 1))
    return list(range(s, 13)) + list(range(1, e + 1))

df["months_open"] = df.apply(months_open, axis=1)
df["months_open_str"] = df["months_open"].apply(lambda m: "|".join(map(str, m)))

df[["Place_Name", "Best_Month_Start", "Best_Month_End", "months_open_str"]].head()

,Place_Name,Best_Month_Start,Best_Month_End,months_open_str
0,Dal Lake,4,10,4|5|6|7|8|9|10
1,Shalimar Bagh,4,10,4|5|6|7|8|9|10
2,Sonamarg,5,9,5|6|7|8|9
3,Gulmarg,3,6,3|4|5|6
4,Noori Chamb Waterfall,4,10,4|5|6|7|8|9|10


In [ ]:
drop_cols = ["Activity_Cost_Max", "HotelcostpernightINR_Max", "FoodcostperdayINR_Max"]
df_out = df.drop(columns=drop_cols)

out_path = os.path.join(PROCESSED_DIR, "places_processed.csv")
df_out.to_csv(out_path, index=False)

print(f"Saved processed table: {out_path}")
print(f"Shape: {df_out.shape[0]} rows, {df_out.shape[1]} columns")
print(f"Artifacts saved in: {ARTIFACTS_DIR}")

Saved processed table: ..\..\data\processed\places_processed.csv
Shape: 1163 rows, 202 columns
Artifacts saved in: ..\artifacts
